# YOLOv10 Training — Blind Cap Assistive Vision

**Run this notebook on Google Colab with a GPU runtime.**

```
Runtime → Change runtime type → Hardware accelerator → GPU (T4 or better)
```

### What this notebook does
1. Verifies GPU is available
2. Installs all dependencies
3. Downloads a subset of Open Images (15 target classes, ~1 400 images)
4. Converts annotations to YOLO format
5. Trains `yolov10n.pt` for 50 epochs
6. Downloads `best.pt` — copy it to `models/best.pt` in your local project

> **Dataset budget:** ~1 200 train + 200 val images ≈ 3–5 GB (well under 10 GB)

## Step 1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'No GPU found. Go to Runtime → Change runtime type → GPU and re-run.'
    )

## Step 2 — Install Dependencies

In [ ]:
!pip install -q ultralytics
!pip install -q oidv6
!pip install -q openimages2yolo

## Step 3 — Create Dataset Directory

In [ ]:
import os
os.makedirs('dataset', exist_ok=True)
os.chdir('dataset')
print('Working directory:', os.getcwd())

## Step 4 — Download Training Images

Downloads ~1 200 images across all 15 target classes (limit is per-run, not per-class).

In [ ]:
!oidv6 downloader \
    --dataset train \
    --type_data images \
    --classes Person Chair Table Sofa Bed Door Bench \
               Car Bus Truck Bicycle Dog \
               "Trash can" "Fire hydrant" Mailbox \
    --limit 1200

In [ ]:
!oidv6 downloader \
    --dataset validation \
    --type_data images \
    --classes Person Chair Table Sofa Bed Door Bench \
               Car Bus Truck Bicycle Dog \
               "Trash can" "Fire hydrant" Mailbox \
    --limit 200

## Step 5 — Download Bounding Box Annotations

In [ ]:
!oidv6 downloader \
    --dataset train \
    --type_data annotations \
    --classes Person Chair Table Sofa Bed Door Bench \
               Car Bus Truck Bicycle Dog \
               "Trash can" "Fire hydrant" Mailbox

In [ ]:
!oidv6 downloader \
    --dataset validation \
    --type_data annotations \
    --classes Person Chair Table Sofa Bed Door Bench \
               Car Bus Truck Bicycle Dog \
               "Trash can" "Fire hydrant" Mailbox

## Step 6 — Convert Annotations to YOLO Format

In [ ]:
os.chdir('..')   # back to /content
!python -m openimages2yolo \
    --input_dir dataset \
    --output_dir dataset_yolo
print('Conversion done.')

## Step 7 — Verify Converted Dataset Structure

Expected layout:
```
dataset_yolo/
  images/
    train/   ← jpg files
    val/     ← jpg files
  labels/
    train/   ← txt files
    val/     ← txt files
```

In [ ]:
import pathlib
for split in ('train', 'val'):
    n_img = len(list(pathlib.Path(f'dataset_yolo/images/{split}').glob('*.jpg')))
    n_lbl = len(list(pathlib.Path(f'dataset_yolo/labels/{split}').glob('*.txt')))
    print(f'{split}: {n_img} images, {n_lbl} labels')

## Step 8 — Write dataset.yaml

In [ ]:
import os
dataset_yaml = f"""\
path: {os.path.abspath('dataset_yolo')}

train: images/train
val: images/val

nc: 15

names:
  0: person
  1: chair
  2: table
  3: sofa
  4: bed
  5: door
  6: bench
  7: trash_can
  8: fire_hydrant
  9: mailbox
  10: car
  11: bus
  12: truck
  13: bicycle
  14: dog
"""

with open('dataset.yaml', 'w') as f:
    f.write(dataset_yaml)

print('dataset.yaml written:')
print(dataset_yaml)

## Step 9 — Train YOLOv10n

- `batch=16` — safe for T4 (16 GB VRAM)
- `device=0` — first GPU
- `epochs=50` — good baseline; increase to 100 for higher mAP

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov10n.pt')

results = model.train(
    data='dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='runs/detect',
    name='blind_cap',
    exist_ok=True,
)

print('Training complete.')
print('Best weights:', results.save_dir)

## Step 10 — Evaluate on Validation Set

In [ ]:
best_pt = 'runs/detect/blind_cap/weights/best.pt'
trained = YOLO(best_pt)
metrics = trained.val(data='dataset.yaml', device=0)
print(f'mAP50   : {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## Step 11 — Download best.pt

After downloading, copy `best.pt` into `models/best.pt` in your local project.

In [ ]:
from google.colab import files
files.download('runs/detect/blind_cap/weights/best.pt')

---
## Next Steps — Local VS Code Project

1. Copy `best.pt` → `blind-cap-object-detection/models/best.pt`
2. Edit `config.yaml` — change `model_path` to `models/best.pt`
3. Validate the model:
   ```bash
   python -m training.train_detector --validate models/best.pt
   ```
4. Run the system:
   ```bash
   python run_app.py --scenario indoor
   ```